In [2]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent

sys.path.append(str(root_dir))


In [3]:
import math
import torch
import numpy as np

class CMAES:
    def __init__(self, x0: torch.Tensor, sigma0: float,
                 popsize: int = None, mu: int = None,
                 device: str = 'cpu', dtype=torch.float64):
        self.device = device
        self.dtype = dtype
        self.N = x0.shape[0]
        
        self.lamb = popsize if popsize else 4 + int(3 * math.log(self.N))
        self.mu = mu if mu else self.lamb // 2
        
        w = torch.tensor([math.log(self.mu + 0.5) - math.log(i + 1)
                          for i in range(self.mu)], device=device, dtype=dtype)
        w /= w.sum()
        self.w = w
        self.mueff = 1.0 / (w ** 2).sum()
        
        self.cc = (4.0 + self.mueff / self.N) / (self.N + 4.0 + 2.0 * self.mueff / self.N)
        self.cs = (self.mueff + 2.0) / (self.N + self.mueff + 5.0)
        self.c1 = 2.0 / ((self.N + 1.3) ** 2 + self.mueff)
        self.cmu = min(1.0 - self.c1,
                       2.0 * (self.mueff - 2.0 + 1.0 / self.mueff) / ((self.N + 2.0) ** 2 + self.mueff))
        self.damps = (1.0 + 2.0 * max(0.0, math.sqrt((self.mueff - 1.0) / (self.N + 1.0)) - 1.0) + self.cs)
        self.chiN = math.sqrt(self.N) * (1.0 - 1.0 / (4.0 * self.N) + 1.0 / (21.0 * self.N ** 2))
        
        self.mean = x0.to(device=device, dtype=dtype)
        self.sigma = torch.tensor(sigma0, device=device, dtype=dtype)
        self.C = torch.eye(self.N, device=device, dtype=dtype)
        self.pc = torch.zeros(self.N, device=device, dtype=dtype)
        self.ps = torch.zeros(self.N, device=device, dtype=dtype)
        
        self._eigen_decomposition()
        
        self.z = None
        self.y = None
        self.x = None
        
        self.generation = 0
        self.best_x = self.mean.clone()
        self.best_f = float('inf')
    
    def _eigen_decomposition(self):
        D, B = torch.linalg.eigh(self.C)
        D = torch.clamp(D, min=1e-10)
        self.B = B
        self.D = D
        self.sqrtD = torch.sqrt(D)
        self.invsqrtD = 1.0 / self.sqrtD
    
    def ask(self) -> torch.Tensor:
        """Возвращает популяцию-кандидатов (lambda, N)."""
        self.z = torch.randn(self.lamb, self.N, device=self.device, dtype=self.dtype)
        self.y = (self.z * self.sqrtD) @ self.B.T
        self.x = self.mean + self.sigma * self.y
        return self.x
    
    def tell(self, fitnesses: torch.Tensor):
        idx = torch.argsort(fitnesses)
        fitnesses = fitnesses[idx]

        # запоминаем лучшее значение текущего поколения
        current_best_f = fitnesses[0].item()
        if current_best_f < self.best_f:
            self.best_f = current_best_f
            self.best_x = self.x[idx[0]].clone()
        
        # Упорядочиваем z, y, x
        z = self.z[idx]
        y = self.y[idx]
        x = self.x[idx]
        
        # Взвешенная рекомбинация (только mu лучших)
        yw = (self.w.unsqueeze(1) * y[:self.mu]).sum(dim=0)   # y_w
        self.mean = self.mean + self.sigma * yw
        
        # Обратный квадратный корень из C, умноженный на yw
        # C^{-1/2} * yw = B * diag(1/sqrt(D)) * B^T * yw
        Bt_yw = self.B.T @ yw
        inv_sqrtC_yw = self.B @ (Bt_yw * self.invsqrtD)
        
        # Обновление эволюционного пути для шага sigma
        self.ps = ((1 - self.cs) * self.ps +
                   math.sqrt(self.cs * (2 - self.cs) * self.mueff) * inv_sqrtC_yw)
        
        # Обновление sigma
        ps_norm = torch.norm(self.ps)
        self.sigma *= torch.exp(self.cs / self.damps * (ps_norm / self.chiN - 1.0)).item()
        
        # Пороговое условие для hsig
        hsig = (ps_norm / math.sqrt(1 - (1 - self.cs) ** (2 * (self.generation + 1)))
                < (1.4 + 2.0 / (self.N + 1.0)) * self.chiN)
        hsig = float(hsig)  # 1.0 или 0.0
        
        # Обновление эволюционного пути для ковариационной матрицы
        self.pc = ((1 - self.cc) * self.pc +
                   hsig * math.sqrt(self.cc * (2 - self.cc) * self.mueff) * yw)
        
        # Ранговое обновление ковариационной матрицы
        # C = (1 - c1 - cmu) * C  +  c1 * (pc*pc^T + (1-hsig)*cc*(2-cc)*C)
        #      + cmu * sum_{i=1}^{mu} w_i * y_i * y_i^T
        pc_outer = torch.outer(self.pc, self.pc)
        rank_mu = torch.zeros_like(self.C)
        for i in range(self.mu):
            rank_mu += self.w[i] * torch.outer(y[i], y[i])
        
        self.C = ((1 - self.c1 - self.cmu) * self.C +
                  self.c1 * (pc_outer + (1 - hsig) * self.cc * (2 - self.cc) * self.C) +
                  self.cmu * rank_mu)
        
        # Обновляем собственное разложение
        self._eigen_decomposition()
        
        self.generation += 1

In [4]:
def quadratic(x):
    """Функция-пример: сумма квадратов (минимум в нуле)."""
    return (x ** 2).sum(dim=1)

torch.manual_seed(42)
dim = 2
x0 = torch.randn(dim) * 5       # начальная точка
cma = CMAES(x0, sigma0=0.5)

for gen in range(200):
    pop = cma.ask()
    fitness = quadratic(pop)
    cma.tell(fitness)
    if gen % 20 == 0:
        print(f"Gen {gen:3d}: f_best = {cma.best_f:.4f}, sigma = {cma.sigma:.4f}")

print("Лучшее решение:", cma.best_x)

Gen   0: f_best = 1.5624, sigma = 0.4392
Gen  20: f_best = 0.0000, sigma = 0.0862
Gen  40: f_best = 0.0000, sigma = 0.0032
Gen  60: f_best = 0.0000, sigma = 0.0002
Gen  80: f_best = 0.0000, sigma = 0.0000
Gen 100: f_best = 0.0000, sigma = 0.0000
Gen 120: f_best = 0.0000, sigma = 0.0000
Gen 140: f_best = 0.0000, sigma = 0.0000
Gen 160: f_best = 0.0000, sigma = 0.0000
Gen 180: f_best = 0.0000, sigma = 0.0000
Лучшее решение: tensor([-4.9225e-20,  1.6075e-19], dtype=torch.float64)


In [5]:
from controller import Controller
from embedder import CNNVAE
from predictor import PredictorTransformer

In [6]:
controller = Controller(64, 3)
vae = CNNVAE(3, 64, 96)
predictor = PredictorTransformer(64, 8, 128, 3, 4, 4, 128)

vae.load_state_dict(torch.load("../embedder/vaev4.pt", map_location='cpu'))
predictor.load_state_dict(torch.load("../predictor/predictor_ml.pt", map_location='cpu'))

vae.eval()
predictor.eval()

PredictorTransformer(
  (act_embedder): Linear(in_features=3, out_features=8, bias=True)
  (in_proj): Linear(in_features=72, out_features=128, bias=True)
  (pe): SinusoidalPositionalEncoding()
  (layers): ModuleList(
    (0-3): 4 x TransformerBlock(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (mha): MultiHeadAttention(
        (q_proj): Linear(in_features=128, out_features=128, bias=True)
        (k_proj): Linear(in_features=128, out_features=128, bias=True)
        (v_proj): Linear(in_features=128, out_features=128, bias=True)
        (out_proj): Linear(in_features=128, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (dropout1): Dropout(p=0.1, inplace=False)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (ffn): SimpleFFN(
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (linear2): Linear(in_features=512, out_features=128, bias=True

In [7]:
import torch
import torch.nn as nn
import gymnasium as gym

In [ ]:
from torch.nn.utils import parameters_to_vector, vector_to_parameters

x0 = parameters_to_vector(controller.parameters())   # плоский тензор (float32 по умолчанию)

# Узнаём общее число параметров
print(f"Размерность задачи (число параметров): {x0.shape[0]}")

# 3. Функция оценки (теперь принимает вектор параметров и восстанавливает модель)
def evaluate_policy(weights, model, env):
    """
    weights: плоский вектор параметров
    model: экземпляр сети (будет перезаписан)
    env: окружение gym
    Возвращает -total_reward
    """
    # Загружаем веса в модель
    vector_to_parameters(weights.to(model_device, dtype=model_dtype), model.parameters())

    obs, _ = env.reset()
    total_reward = 0.0
    done = False
    actions_list = [torch.tensor(np.array([0, 0, 0], dtype=np.float32))]
    observations_list = []
    st = 0
    while st < 900 or not done:
        with torch.no_grad():
            obs_tensor = torch.from_numpy(obs/255).unsqueeze(0).to(model_device, dtype=model_dtype)
            
            z_now = vae.reparameterize(*vae.encode(obs_tensor.view(-1, 3, 96, 96)))
            observations_list.append(z_now)
            if len(observations_list) >127:
                actions_list = actions_list[1:]
                observations_list = observations_list[1:]
            

            A = torch.stack(actions_list).unsqueeze(0).to(torch.float32)
            Z = torch.stack(observations_list).permute(1, 0, 2)
            mu, logvar = predictor(Z, A)
            mu = mu[:, -1, :]
            logvar = logvar[:, -1, :]
            std = torch.exp(0.5*logvar)
            eps=torch.rand_like(std)
            z_next = mu+eps*std
            logits = model(z_now, z_next)
            action = logits.detach().cpu()
            actions_list.append(action[0])
        obs, reward, terminated, truncated, _ = env.step(action[0].numpy())
        total_reward += reward
        done = terminated or truncated
        st = st +1
    return -total_reward

# 4. Настройка CMA-ES
device = "cpu"            # можно "cuda", но CMA-ES с нейросетями дорог; для простоты CPU
model_device = device
model_dtype = torch.float32   # сеть обычно float32

# Параметры CMA-ES: популяция побольше, sigma0 на ваш выбор
cma = CMAES(x0.to(device=device, dtype=torch.float64),   # CMAES внутри float64
            sigma0=0.5,
            popsize=50,
            device=device)

# Для оценки нужно отдельное окружение (train), чтобы не мешать визуализации
env = gym.make(
            "CarRacing-v3",
            render_mode="rgb_array",
            lap_complete_percent=0.95,
            domain_randomize=False,
            continuous=True               # <-- теперь можно одновременно делать несколько действий
        )

# Обучение
generations = 100
for gen in range(generations):
    pop = cma.ask()   # float64
    fitness = torch.tensor([
        evaluate_policy(pop[i].float(), controller, env)   # приводим к float32
        for i in range(pop.shape[0])
    ], device=device, dtype=torch.float64)
    cma.tell(fitness)
    print(f"Поколение {gen:3d}: лучшая награда = {-cma.best_f:.1f}")

env.close()

Размерность задачи (число параметров): 387
Поколение   0: лучшая награда = -59.8
Поколение   1: лучшая награда = -57.4
Поколение   2: лучшая награда = -50.0
Поколение   3: лучшая награда = -50.0
Поколение   4: лучшая награда = -50.0
Поколение   5: лучшая награда = -50.0
Поколение   6: лучшая награда = -49.1
Поколение   7: лучшая награда = -14.9
Поколение   8: лучшая награда = -14.9
Поколение   9: лучшая награда = -14.9
Поколение  10: лучшая награда = -14.9
Поколение  11: лучшая награда = -4.4
Поколение  12: лучшая награда = -4.4
Поколение  13: лучшая награда = -4.4
Поколение  14: лучшая награда = -4.4
Поколение  15: лучшая награда = -4.4
Поколение  16: лучшая награда = -4.4
Поколение  17: лучшая награда = -4.4
Поколение  18: лучшая награда = -4.4


In [31]:

# 5. Визуализация лучшей политики
best_weights = cma.best_x.to(device="cpu", dtype=torch.float32)
vector_to_parameters(best_weights, model.parameters())   # загружаем лучшие веса

env_vis = gym.make("CartPole-v1", render_mode="human")
obs, _ = env_vis.reset()
total_reward = 0.0
done = False
while not done:
    obs_tensor = torch.from_numpy(obs).unsqueeze(0).to(torch.float32)
    with torch.no_grad():
        action = torch.argmax(model(obs_tensor), dim=1).item()
    obs, reward, terminated, truncated, _ = env_vis.step(action)
    total_reward += reward
    done = terminated or truncated
print(f"Визуализация. Общая награда: {total_reward}")
env_vis.close()

Визуализация. Общая награда: 500.0


In [ ]:
env.step()


(array([[[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        ...,
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]]], shape=(96, 96, 3), dtype=uint8),
 8.13045267489712,
 False,
 False,
 {})